#### Agent（智能体或代理）
##### 1. 什么是Agent
什么是Agent，这其实没有一个标准的答案，每个人都有自己的理解。
> 对于这个问题，LangChain创始人Harrison Chase有一个偏向技术性的答案：
>
> An AI agent is a system that uses an LLM to decide the control flow of an application.
>
> Agent是一种使用大语言模型（LLM）来决定应用程序控制流的系统。
在人工智能领域，Agent（智能体或代理）是指一种能够感知环境、进行推理、自主决策并采取行动心实现特定目标智能系统。
1.人物规划 -> 2.调用工具 -> 3.感知反馈

总结如下：
• LLM=聪明的大脑
• Agent=聪明的大脑+工具（手脚/函数）
##### 2. 工具调用（ToolCall）原理
```python
def get_weather(location: str) -> str:
    """Get the weather in a given location.

    Args:
        location: city name or coordinates

    Returns:
        weather: weather description
    """
    return f"Current weather in {location} is sunny"
```

*文档字符串规范（Docstrings — PEP 257 & Google 风格）*
```python
async def query_product_page(
    self,
    category: str | None = None,
    product_name: str | None = None,
    page: int = 1,
    page_size: int = 10,
) -> tuple[list[Product], int]:
    """分页动态组合查询保险产品列表。

    根据分类过滤条件和产品名称关键词进行模糊匹配，支持分页。
    默认仅查询状态为 active 的上架产品。

    Args:
        category: 产品分类代码，如 medical, life 等
        product_name: 产品名称模糊搜索关键词
        page: 当前页码，从 1 开始
        page_size: 每页显示条数，最大限制 100

    Returns:
        tuple[list[Product], int]: 包含两个元素的元组：
            - 第一个元素为当前页的产品数据库模型列表
            - 第二个元素为符合条件的总记录数 (total)

    Raises:
        ValueError: 当 page 小于 1 或 page_size 超出合理范围时
    """
```
![image.png](attachment:465b5380-0d43-48e6-a90b-030ae89c3103.png)

- DeepSeek API 文档
  - Chat Completions API：https://api-docs.deepseek.com/zh-cn/api/create-chat-completion/
##### 3. 手写ToolCall

In [1]:
import json
import os

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()


# ==================== 1. 定义工具 ====================
def get_weather(location: str) -> str:
    return f"Current weather in {location} is sunny"


def square_root(x: float) -> float:
    return x**0.5


# 把函数放到dict中，方便根据函数名查找
tools_by_name = {"get_weather": get_weather, "square_root": square_root}

# ==================== 2. 定义工具的LLM描述 ====================
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get weather of a location, the user should supply a location first.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA",
                    }
                },
                "required": ["location"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "square_root",
            "description": "Calculate the square root of a given number.",
            "parameters": {
                "type": "object",
                "properties": {
                    "x": {
                        "type": "number",
                        "description": "The number to calculate square root for",
                    }
                },
                "required": ["x"],
            },
        },
    },
]

# ==================== 3. LLM客户端 ====================
client = OpenAI(api_key=os.getenv("DEEPSEEK_API_KEY"), base_url="https://api.deepseek.com")


# ==================== 4. 定义发送消息的方法 ====================
def send_messages(messages):
    response = client.chat.completions.create(
        model="deepseek-v4-flash",
        messages=messages,
        tools=tools,
        extra_body={"thinking": {"type": "disabled"}},
    )
    return response.choices[0].message


# ==================== 5. 定义执行Tool调用的方法 ====================
def execute_tool(message, messages):
    """解析Tool信息，执行对应的函数，并将结果追加到消息列表"""
    tool_call = message.tool_calls[0]
    tool_id = tool_call.id
    tool_name = tool_call.function.name
    tool_args = json.loads(tool_call.function.arguments)

    print(f"调用工具: {tool_name}, 参数: {tool_args}")

    # 根据函数名找到对应函数并执行
    result = tools_by_name[tool_name](**tool_args)
    print(f"工具结果: {result}")

    # 将AI消息和Tool结果追加到消息列表
    messages.append(message)
    messages.append({"role": "tool", "tool_call_id": tool_id, "content": result})
    return messages


# ==================== 6. 主流程 ====================
# 准备用户消息
messages = [{"role": "user", "content": "杭州天气怎么样?"}]
print(f"User> {messages[0]['content']}")

# 第一次调用LLM，获取ToolCall
message = send_messages(messages)
print(f"Tool> {message.tool_calls[0]}")

# 执行Tool并将结果传回LLM
messages = execute_tool(message, messages)

# 再次调用LLM，生成最终答案
message = send_messages(messages)
print(f"AI> {message.content}")

User> 杭州天气怎么样?
Tool> ChatCompletionMessageFunctionToolCall(id='call_00_wLlBXYAwki3iFEfSEqzO6401', function=Function(arguments='{"location": "杭州"}', name='get_weather'), type='function', index=0)
调用工具: get_weather, 参数: {'location': '杭州'}
工具结果: Current weather in 杭州 is sunny
AI> 杭州目前天气**晴朗**（sunny）☀️。

不过这个查询只返回了天气状况，没有提供具体温度和湿度等详细信息。如果您需要更详细的数据（比如气温、风力、空气质量等），建议查看当地天气预报应用。


![image.png](attachment:ddb82399-59f0-4836-8cce-cc799a7a2578.png)
流程如下：
1. 用户提问（Input）：杭州今天天气如何？
2. 模型分析（Reasoning）：用户询问杭州天气，我不知道，需要调用查询天气的工具 get_weather
3. 调用工具（Action）：调用工具，get_weather，传入城市“杭州”
4. 分析结果（Observation）：工具返回结果，模型分析结果，判断是否足以回答用户问题
    a. 是：整理生成响应结果
    b. 否：重复前面步骤
5. 生成结果（Output）：根据工具的结果生成响应给用户
上图所说的这个工作模型，我们称为ReAct模型，即Reasoning Action，而这也是最基础一种Agent工作模型。
恭喜你！已经完成了一个简单的Agent了